# FFMitra Model Evaluation

Loads the trained artifacts, recomputes test metrics, and plots ROC / PR curves and the confusion matrix. Run `backend/scripts/train.py` first to produce the artifacts.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

ROOT = Path.cwd().parent if Path.cwd().name == "backend" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

from app.ml.features import FEATURES, build_training_frame
from sklearn.model_selection import train_test_split

MODELS_DIR = ROOT / "data" / "models"
DATASETS_DIR = ROOT / "data" / "datasets"

In [ ]:
model = joblib.load(MODELS_DIR / "fraud_xgb.joblib")
iso = joblib.load(MODELS_DIR / "isoforest.joblib")
meta = json.loads((MODELS_DIR / "feature_metadata.json").read_text(encoding="utf-8"))
print("artifacts loaded; trained_at:", meta["trained_at"])
print("stored metrics:", {k: v for k, v in meta["metrics"].items() if k != "feature_importance"})

In [ ]:
frames = []
if (DATASETS_DIR / "creditcard.csv").exists():
    card = pd.read_csv(DATASETS_DIR / "creditcard.csv")
    card = card.sample(frac=0.35, random_state=42).reset_index(drop=True)
    from datetime import datetime, timedelta

    card["source_ref"] = [f"CARD_{i % 50}" for i in range(len(card))]
    card["dest_ref"] = "CARD_DEST_0"
    card["device_id"] = [f"CARD_DEVICE_{i % 7}" for i in range(len(card))]
    card["location"] = [f"LOC_{i % 3}" for i in range(len(card))]
    card["account_age_days"] = 365
    card["expected_daily"] = 10.0
    card["cross_bank"] = 0
    card["geo"] = 0.0
    card["channel"] = "CARD"
    card["txn_time"] = card["Time"].map(
        lambda s: (datetime(2025, 1, 1) + timedelta(seconds=float(s))).isoformat()
    )
    card["is_fraud"] = card["Class"].astype(int)
    frames.append(
        build_training_frame(
            card[["source_ref", "dest_ref", "amount", "txn_time", "device_id", "location", "account_age_days", "is_fraud"]]
        )
    )
frames.append(build_training_frame(pd.read_csv(DATASETS_DIR / "synthetic_transactions.csv")))
frame = pd.concat(frames, ignore_index=True)
print("eval frame:", frame.shape)

y = frame["is_fraud"].values
X = frame[FEATURES].values.astype(np.float64)
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("test rows:", len(y_test))

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
cm = confusion_matrix(y_test, (y_proba >= 0.5).astype(int))
print(f"ROC AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f}")
print(f"confusion matrix:\n{cm}")

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
precision, recall, _ = precision_recall_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(fpr, tpr, label=f"XGB (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].plot(recall, precision, label=f"XGB (PR AUC = {pr_auc:.3f})")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

im = axes[2].imshow(cm, cmap="Blues")
axes[2].set_title("Confusion Matrix (thr=0.5)")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black")
axes[2].set_xticks([0, 1])
axes[2].set_yticks([0, 1])
fig.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
importances = dict(zip(FEATURES, model.feature_importances_))
top = sorted(importances.items(), key=lambda kv: kv[1], reverse=True)[:12]
plt.figure(figsize=(9, 5))
plt.barh([f[0] for f in top][::-1], [f[1] for f in top][::-1], color="steelblue")
plt.title("Top Feature Importances (XGB)")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

In [ ]:
anomaly = iso.score_samples(X_test)
low, high = -0.4, 0.95
norm = np.clip((anomaly - low) / (high - low), 0.0, 1.0)
print(f"isoforest raw scores: min {anomaly.min():.3f}, max {anomaly.max():.3f}")
print(f"normalized anomaly scores: mean {norm.mean():.3f}, std {norm.std():.3f}")
print(f"anomaly score correlation with fraud label: {np.corrcoef(norm, y_test)[0, 1]:.3f}")